Assignment_1_Pytorch_MNIST.ipynb

Automatically generated by Colaboratory.

Overall structure:

1) Set Pytorch metada
- seed
- tensorflow output
- whether to transfer to gpu (cuda)

2) Import data
- download data
- create data loaders with batchsize, transforms, scaling

3) Define Model architecture, loss and optimizer

4) Define Test and Training loop
    - Train:
        a. get next batch
        b. forward pass through model
        c. calculate loss
        d. backward pass from loss (calculates the gradient for each parameter)
        e. optimizer: performs weight updates

5) Perform Training over multiple epochs:
    Each epoch:
    - call train loop
    - call test loop

Acknowledgments:https://github.com/motokimura/pytorch_tensorboard/blob/master/main.py

Import Statements

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.tensorboard
import os
import torchvision
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from pathlib import Path
from torch.utils.data import DataLoader

In [ ]:
#!pip install cuda-python
#!pip install tensorboard
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
batch_size = 64
val_batch_size = 1000
epochs = 10
lr = 0.01
try_cuda = True
seed = 1000
logging_interval = 100 # how many batches to wait before logging
logging_dir = None

In [ ]:
# 1) setting up the logging
writer = SummaryWriter()

In [ ]:
#deciding whether to send to the cpu or not if available
if torch.cuda.is_available() and try_cuda:
    cuda = True
    torch.cuda.manual_seed(seed)
else:
    cuda = False
    torch.manual_seed(seed)

In [ ]:
# Setting up data
transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

In [ ]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=2)

In [ ]:
# Defining Architecture,loss and optimizer
class Net(nn.Module):
    
    def __init__(self):

        super(Net, self).__init__()
        # 1 input channel (black/white image), 32 output channels, 5x5 square convolution
        self.conv1 = nn.Conv2d(1, 32, 5)
        # 32 input channels, 64 output channels, 5x5 square convolution
        self.conv2 = nn.Conv2d(32, 64, 5)
        
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()
        self.relu3 = nn.ReLU()

        self.pool1 = nn.MaxPool2d(2)
        self.pool2 = nn.MaxPool2d(2)

        self.dropout1 = nn.Dropout(0.5)       
        
        self.fc1 = nn.Linear(1024, 10)
        nn.init.xavier_uniform_(self.fc1.weight)
        
    def forward(self, x):
        
        x = self.conv1(x)
        x = self.relu1(x)

        x = self.pool1(x)
        x = self.conv2(x)

        x = self.relu2(x)
        x = self.pool2(x)

        x = torch.flatten(x, 1) # flatten feature map to [batch_size, 1024]
        x = self.fc1(x)

#         x = self.relu3(x)
#         x = self.dropout1(x)
        
#         x = F.softmax(x,dim=1)

        return x

In [ ]:
# instantiate model
model = Net()

if cuda:
    model = model.cuda()

In [ ]:
# Dictionary that stores outputs from selected layers
activation_data = {}

def save_output(name):
    def hook(module, input, output):
        activation_data[name] = output.detach()
    return hook


# Net inputs / pre-activations
model.conv1.register_forward_hook(save_output("conv1_net"))
model.conv2.register_forward_hook(save_output("conv2_net"))
model.fc1.register_forward_hook(save_output("fc1_net"))

# Activations after ReLU
model.relu1.register_forward_hook(save_output("relu1"))
model.relu2.register_forward_hook(save_output("relu2"))

# Activations after Max-Pooling
model.pool1.register_forward_hook(save_output("pool1"))
model.pool2.register_forward_hook(save_output("pool2"))

In [ ]:
# loss function
criterion = nn.CrossEntropyLoss()

## USE AN ADAM OPTIMIZER
# optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = optim.SGD(model.parameters(), lr=0.001)

In [ ]:
# Defining the validation and training loops
eps=1e-13

In [ ]:
def log_tensor_stats(epoch, writer, tag, tensor):
    tensor = tensor.detach().float().cpu()

    writer.add_scalar(f"{tag}/min", tensor.min().item(), epoch)
    writer.add_scalar(f"{tag}/max", tensor.max().item(), epoch)
    writer.add_scalar(f"{tag}/mean", tensor.mean().item(), epoch)
    writer.add_scalar(f"{tag}/std", tensor.std().item(), epoch)

    writer.add_histogram(
        f"{tag}/histogram",
        tensor,
        epoch
    )

In [ ]:
# Use the same batch every epoch for monitoring
monitor_data, monitor_target = next(iter(val_loader))

if cuda:
    monitor_data = monitor_data.cuda()
    monitor_target = monitor_target.cuda()

In [ ]:
def log_activations(epoch, writer):

    model.eval()

    with torch.no_grad():
        _ = model(monitor_data)

    # Net inputs / pre-activations
    log_tensor_stats(epoch, writer, "NetInputs/conv1", activation_data["conv1_net"])
    log_tensor_stats(epoch, writer, "NetInputs/conv2", activation_data["conv2_net"])
    log_tensor_stats(epoch, writer, "NetInputs/fc1", activation_data["fc1_net"])

    # ReLU activations
    log_tensor_stats(epoch, writer, "Activations/ReLU1", activation_data["relu1"])
    log_tensor_stats(epoch, writer, "Activations/ReLU2", activation_data["relu2"])

    # Max-pooling activations
    log_tensor_stats(epoch, writer, "Activations/MaxPool1", activation_data["pool1"])
    log_tensor_stats(epoch, writer, "Activations/MaxPool2", activation_data["pool2"])

In [ ]:
def train(epoch, writer):
    model.train()
    running_loss = 0.
    last_loss = 0.
    
    # Here, we use enumerate(training_loader) instead of
    # iter(training_loader) so that we can track the batch
    # index and do some intra-epoch reporting
    for batch_idx, (data, target) in enumerate(train_loader):
        if cuda:
            data, target = data.cuda(), target.cuda()
            
        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        output = model(data)

        # Compute the loss and its gradients
        loss = criterion(output, target)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # Gather data and report
        running_loss += loss.item()
        
        if batch_idx % logging_interval == 0:
            last_loss = running_loss / logging_interval # loss per batch
            tb_x = epoch * len(train_loader) + batch_idx + 1
            writer.add_scalar('Loss/train', last_loss, tb_x)
            print('Train set: Epoch: {} Batch: {} Loss: {:.4f}\n'.format(epoch, batch_idx + 1, last_loss))
            running_loss = 0. # reset running loss calculation
    
    # Log model parameters to TensorBoard at every epoch
    for name, param in model.named_parameters():
      if "weight" in name:
          tag = f"Weights/{name}"
      elif "bias" in name:
          tag = f"Biases/{name}"
      else:
          continue
      log_tensor_stats(epoch, writer, tag, param)

In [ ]:
def validate(epoch, writer):
    model.eval()
    val_loss = 0
    correct = 0

    for data, target in val_loader:
        if cuda:
            data, target = data.cuda(), target.cuda()

        output = model(data)
        
        val_loss += criterion(output, target).item() # sum up batch loss (later, averaged over all validation samples)
        pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
        correct += pred.eq(target.view_as(pred)).sum().item()

    val_loss /= len(val_loader.dataset)
    val_accuracy = 100. * correct / len(val_loader.dataset)
    print('Validation set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(val_loss, correct, len(val_loader.dataset), val_accuracy))

#     Log val/loss and val/accuracy to TensorBoard at every epoch
    n_iter = epoch * len(train_loader)
    writer.add_scalar('Loss/validation', val_loss, n_iter)
    writer.add_scalar('Accuracy/validation', val_accuracy, n_iter)

In [ ]:
# Training loop
for epoch in range(epochs):
    train(epoch, writer)
    validate(epoch, writer)
    log_activations(epoch, writer)
writer.flush()

In [ ]:
writer.close()